In [ ]:
# ============================================================
# CELL 1 — INSTALLATION
# Google ADK replaces LangGraph as the orchestration layer.
# Everything else (Docling, Whisper, ChromaDB, Neo4j) stays.
#
# WHY ADK INSTEAD OF LANGGRAPH?
#   LangGraph: you manually wire nodes, edges, state dicts.
#   ADK:       you define Agents + Tools as Python functions.
#              ADK handles routing, retries, streaming, sessions,
#              and a built-in web debug UI out of the box.
#              ADK 2.0 adds graph-based Workflow orchestration
#              that maps perfectly to our ingestion pipeline.
# ============================================================

!pip install -q \
    google-adk \
    google-genai \
    docling \
    chromadb \
    neo4j \
    diskcache \
    openai-whisper \
    opencv-python-headless \
    filetype \
    jsonlines \
    openpyxl \
    pandas \
    Pillow \
    python-dotenv \
    sqlalchemy \
    nest_asyncio

# ADK needs an event loop — nest_asyncio lets it run inside Colab
import nest_asyncio
nest_asyncio.apply()

print("✅ All packages installed.")
print("   google-adk is now the orchestration layer.")
print("   LangGraph is no longer used.")


In [ ]:
# ============================================================
# CELL 2 — SECRETS & ENVIRONMENT
#
# ADK difference: ADK reads GOOGLE_API_KEY from environment
# automatically. We still use .env on Drive for all secrets
# but we now set GOOGLE_API_KEY (ADK's expected var name)
# instead of passing the key manually to genai.Client().
# ============================================================

import os
from pathlib import Path
from dotenv import load_dotenv

# ── Mount Google Drive ───────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/idp_adk")
except ImportError:
    DRIVE_ROOT = Path("./idp_adk")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# ── Create .env if missing ───────────────────────────────────
ENV_FILE = DRIVE_ROOT / ".env"
if not ENV_FILE.exists():
    ENV_FILE.write_text(
        "# ADK reads GOOGLE_API_KEY automatically\n"
        "GOOGLE_API_KEY=YOUR_GEMINI_KEY_HERE\n"
        "NEO4J_URI=neo4j+s://YOUR_SANDBOX.databases.neo4j.io\n"
        "NEO4J_USER=neo4j\n"
        "NEO4J_PASSWORD=YOUR_NEO4J_PASSWORD\n"
    )
    raise EnvironmentError(
        f"Created {ENV_FILE} — fill in your keys and re-run this cell."
    )

load_dotenv(ENV_FILE)

# ADK picks this up automatically — no manual client init needed
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
NEO4J_URI      = os.getenv("NEO4J_URI", "")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")

if "YOUR_" in GOOGLE_API_KEY or not GOOGLE_API_KEY:
    raise EnvironmentError("GOOGLE_API_KEY not set. Edit your .env on Drive.")

# Also set explicitly in env so ADK's internal runner finds it
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print(f"✅ Secrets loaded.")
print(f"   API key : {'*' * (len(GOOGLE_API_KEY)-4)}{GOOGLE_API_KEY[-4:]}")
print(f"   Neo4j   : {NEO4J_URI[:35]}…" if NEO4J_URI else "   Neo4j   : not set")

# ── Workspace directories ────────────────────────────────────
LOCAL_WS     = Path("/content/local_idp_runtime")
CHROMA_DIR   = LOCAL_WS / "chroma_db"
IMAGE_DIR    = LOCAL_WS / "extracted_images"
AUDIO_DIR    = LOCAL_WS / "extracted_audio"
FRAME_DIR    = LOCAL_WS / "extracted_frames"

for p in [CHROMA_DIR, IMAGE_DIR, AUDIO_DIR, FRAME_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE  = DRIVE_ROOT / "ingestion_manifest.json"
CHROMA_BACKUP  = DRIVE_ROOT / "chroma_db_backup.zip"
VALIDATION_LOG = DRIVE_ROOT / "validation.jsonl"
INPUT_DIR      = DRIVE_ROOT / "input"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_MODEL = "gemini-2.5-flash"

print(f"\n✅ Workspace ready. Drop input files into:\n   {INPUT_DIR}")


In [ ]:
# ============================================================
# CELL 3 — IMPORTS & SHARED STATE
#
# ADK difference: we no longer import from langgraph.
# We import from google.adk:
#   Agent    — an LLM-backed agent with tools
#   Workflow — a graph-based deterministic pipeline (ADK 2.0)
#
# The key ADK mental model:
#   Tool     = a Python function the agent can call
#   Agent    = LLM + instruction + list of tools
#   Workflow = deterministic graph of Agent nodes
# ============================================================

import io, re, json, time, zipfile, queue, hashlib
import os, threading, logging as _logging
from typing import List, Dict, Any, Optional
from pathlib import Path

import pandas as pd
import filetype
import jsonlines
import diskcache
import cv2
import whisper
from PIL import Image as PILImage

# Suppress noise
os.environ["TOKENIZERS_PARALLELISM"]      = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"]      = "error"

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

for _lib in ["huggingface_hub", "transformers", "sentence_transformers",
             "chromadb", "urllib3", "httpx", "neo4j", "neo4j.notifications"]:
    _logging.getLogger(_lib).setLevel(_logging.ERROR)

# ── Google ADK ───────────────────────────────────────────────
from google.adk.agents import Agent
# Workflow is ADK 2.0's graph-based pipeline orchestrator
# It replaces LangGraph's StateGraph entirely
try:
    from google.adk import Workflow
    ADK_WORKFLOW_AVAILABLE = True
    print("[ADK] ✅ Workflow (ADK 2.0) available.")
except ImportError:
    ADK_WORKFLOW_AVAILABLE = False
    print("[ADK] ⚠️  Workflow not available — using ADK 1.x SequentialAgent fallback.")
    from google.adk.agents import SequentialAgent

# ── Gemini client (still used for direct API calls in tools) ─
from google import genai
from google.genai import types as genai_types
_gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

# ── Docling ──────────────────────────────────────────────────
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import PictureItem

_docling_opts = PdfPipelineOptions()
_docling_opts.generate_picture_images = True
DOCLING_CONVERTER = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=_docling_opts)}
)

# ── Vector store ─────────────────────────────────────────────
import chromadb
from chromadb.utils import embedding_functions

print("[Embeddings] Pre-loading all-MiniLM-L6-v2…")
_EMBED_FN = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
_ = _EMBED_FN(["warmup"])
print("[Embeddings] ✅ Ready.")

def get_chroma_collections():
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    child_col  = client.get_or_create_collection("child_vectors",  embedding_function=_EMBED_FN)
    parent_col = client.get_or_create_collection("parent_blocks",  embedding_function=_EMBED_FN)
    return child_col, parent_col

# ── Graph store ──────────────────────────────────────────────
from neo4j import GraphDatabase

# ── Whisper ──────────────────────────────────────────────────
print("[Whisper] Loading base model…")
WHISPER_MODEL = whisper.load_model("base")
print("[Whisper] ✅ Ready.")

# ── Query cache ──────────────────────────────────────────────
QUERY_CACHE = diskcache.Cache(str(LOCAL_WS / "query_cache"))

# ── Manifest ─────────────────────────────────────────────────
GLOBAL_MANIFEST: Dict[str, dict] = {}
if MANIFEST_FILE.exists():
    try:
        GLOBAL_MANIFEST = json.loads(MANIFEST_FILE.read_text())
        print(f"[Manifest] Loaded {len(GLOBAL_MANIFEST)} previously processed files.")
    except Exception:
        pass

# ── Ingestion queue + dead letter ────────────────────────────
INGESTION_QUEUE   = queue.Queue()
DEAD_LETTER_QUEUE: List[dict] = []

# ── Rate-limit gate (shared across all Gemini callers) ───────
# This is the core fix for 429 exhaustion.
# All tool functions that call Gemini go through llm_call().
GRAPH_EXTRACTION_ENABLED = True   # paused during ask()

print("\n✅ Cell 3 complete.")


In [ ]:
# ============================================================
# CELL 4 — VALIDATION, LOGGING & STORAGE HELPERS
# Unchanged from v2 — these are infrastructure, not ADK-specific.
# ============================================================

import zipfile

# ── Structured JSONL logger ──────────────────────────────────
def log_event(level: str, file_name: str, element_type: str,
              message: str, extra: dict = None):
    entry = {
        "ts":    time.strftime("%Y-%m-%dT%H:%M:%S"),
        "level": level, "file": file_name,
        "type":  element_type, "msg": message,
    }
    if extra:
        entry.update(extra)
    try:
        with jsonlines.open(str(VALIDATION_LOG), mode="a") as w:
            w.write(entry)
    except Exception:
        pass
    if level in ("FAILURE", "WARNING"):
        icon = "🛑" if level == "FAILURE" else "⚠️ "
        print(f"   {icon} [{level}] {file_name} | {element_type} | {message}")


# ── Validation Gatekeeper ────────────────────────────────────
class ValidationGatekeeper:
    @staticmethod
    def text(text: str, source: str) -> bool:
        c = text.strip()
        if not c or c.count("\x00") > 2: return False
        if len(c) < 10 and not any(ch.isalnum() for ch in c): return False
        return True

    @staticmethod
    def visual_summary(desc: str, source: str, ref: str) -> bool:
        n = desc.lower().strip()
        if not n: return False
        blocked = ["internal error","cannot analyze","unable to see",
                   "safety block","unsuitable","i can't help"]
        if any(k in n for k in blocked):
            log_event("FAILURE", source, "VLM_SUMMARY",
                      f"Blocked: '{desc[:60]}'"); return False
        if len(n) < 30:
            log_event("WARNING", source, "VLM_SUMMARY",
                      f"Too short ({len(n)} chars)"); return False
        return True

    @staticmethod
    def graph_triples(triples: list, source: str) -> bool:
        if not isinstance(triples, list) or not triples: return False
        req = {"source", "type", "target"}
        for t in triples:
            if not isinstance(t, dict) or not req.issubset(t.keys()):
                log_event("WARNING", source, "GRAPH",
                          f"Malformed triple: {t}"); return False
        return True

    @staticmethod
    def transcript(text: str, source: str) -> bool:
        s = text.strip()
        noise = ["[music]","[silence]","[noise]","(music)","♪"]
        if not s or s.lower() in noise or len(s) < 20: return False
        return True


# ── Storage helpers ──────────────────────────────────────────
def save_manifest():
    try:
        MANIFEST_FILE.write_text(json.dumps(GLOBAL_MANIFEST, indent=2))
    except Exception as e:
        log_event("WARNING", "system", "MANIFEST", str(e))

def backup_chroma():
    print("[Backup] Archiving ChromaDB to Drive…")
    try:
        with zipfile.ZipFile(CHROMA_BACKUP, "w", zipfile.ZIP_DEFLATED) as zf:
            for fp in CHROMA_DIR.rglob("*"):
                if fp.is_file():
                    zf.write(fp, fp.relative_to(LOCAL_WS))
        print(f"[Backup] ✅ {CHROMA_BACKUP.name}")
    except Exception as e:
        log_event("WARNING", "system", "BACKUP", str(e))

def restore_chroma():
    if CHROMA_BACKUP.exists():
        print("[Restore] Syncing ChromaDB from Drive…")
        try:
            with zipfile.ZipFile(CHROMA_BACKUP, "r") as zf:
                zf.extractall(LOCAL_WS)
            print("[Restore] ✅ Done.")
        except Exception as e:
            log_event("WARNING", "system", "RESTORE", str(e))
    else:
        print("[Restore] No backup found — starting fresh.")

def optimise_image(pil_img, max_dim=1200):
    img = pil_img.convert("L")
    if max(img.size) > max_dim:
        scale = max_dim / max(img.size)
        img = img.resize((int(img.size[0]*scale), int(img.size[1]*scale)),
                         PILImage.Resampling.LANCZOS)
    return img

def file_hash(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

def detect_mime(path: str) -> str:
    kind = filetype.guess(path)
    if kind: return kind.mime
    ext = Path(path).suffix.lower()
    return {
        ".pdf":  "application/pdf",
        ".docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        ".pptx": "application/vnd.openxmlformats-officedocument.presentationml.presentation",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".csv":  "text/csv", ".txt": "text/plain",
        ".mp3":  "audio/mpeg", ".wav": "audio/wav", ".m4a": "audio/mp4",
        ".mp4":  "video/mp4", ".mov": "video/quicktime",
        ".png":  "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
    }.get(ext, "application/octet-stream")

def llm_call(contents, system: str = None, max_retries: int = 6) -> Any:
    """
    Rate-limited Gemini call used by ALL tool functions.
    Single entry point = single rate-limit surface.
    Adds 2s inter-call floor to stay under 20 RPM free tier.
    """
    time.sleep(2)   # Floor: 2 s between every call = max 30/min
    config = (genai_types.GenerateContentConfig(system_instruction=system)
              if system else None)
    delay = 8
    for attempt in range(max_retries):
        try:
            return _gemini_client.models.generate_content(
                model=TARGET_MODEL, contents=contents, config=config)
        except Exception as e:
            err = str(e).lower()
            if any(c in err for c in ["429","503","resource_exhausted","unavailable"]):
                print(f"   [Retry {attempt+1}/{max_retries}] Rate limited — {delay}s…")
                time.sleep(delay)
                delay = min(delay * 2, 120)
            else:
                raise
    raise RuntimeError("Gemini unreachable after max retries.")

def chunk_text(text: str, child_size: int = 40) -> List[str]:
    words = text.split()
    if len(words) <= child_size: return [text]
    overlap, step = 8, child_size - 8
    return [" ".join(words[i:i+child_size])
            for i in range(0, len(words), step)
            if len(" ".join(words[i:i+child_size]).strip()) > 10]

print("✅ Cell 4 complete — validation, logging, helpers ready.")


In [ ]:
# ============================================================
# CELL 5 — ADK TOOL FUNCTIONS (INGESTION SIDE)
#
# THIS IS WHERE ADK CHANGES THE ARCHITECTURE FUNDAMENTALLY.
#
# In LangGraph: you defined "node functions" that took a state
# dict and returned a modified state dict. The graph wired them.
#
# In ADK: you define plain Python functions decorated or passed
# as "tools". An Agent decides which tool to call based on the
# function name + docstring. The docstring IS the tool spec —
# write it clearly because the LLM reads it to decide routing.
#
# For ingestion we use tools differently: our IngestionWorkflow
# (a deterministic ADK Workflow) calls these functions directly
# as pipeline steps, not through LLM decision-making. This is
# the right pattern for deterministic ETL pipelines.
#
# Tool function rules in ADK:
#   • Must have type-annotated parameters
#   • Must return a dict  (ADK serialises tool outputs as JSON)
#   • Docstring = tool description seen by the agent LLM
#   • Raise exceptions normally — ADK captures them
# ============================================================

# ────────────────────────────────────────────────────────────
# TOOL: parse_document
# ────────────────────────────────────────────────────────────
def parse_document(file_path: str) -> dict:
    """
    Parse a PDF, DOCX, or PPTX file using Docling.
    Extracts text blocks with page numbers and embedded images
    with VLM descriptions. Returns a list of content chunks.
    """
    path = Path(file_path)
    chunks = []
    result = DOCLING_CONVERTER.convert(path)
    img_counter = 0

    for element, _ in result.document.iterate_items():
        if hasattr(element, "text") and element.text.strip():
            txt = element.text.strip()
            if not ValidationGatekeeper.text(txt, path.name):
                continue
            page = getattr(element, "page_no", None)
            chunks.append({
                "text": txt, "source_file": path.name,
                "source_type": "document",
                "page_or_ts": f"page {page}" if page else "n/a",
                "image_path": "NONE",
            })
        elif isinstance(element, PictureItem):
            img_counter += 1
            raw_pil = element.get_image(result.document)
            if not raw_pil: continue
            compressed = optimise_image(raw_pil)
            img_path = IMAGE_DIR / f"{path.stem}_{img_counter}.png"
            compressed.save(img_path, "PNG")
            try:
                resp = llm_call([
                    "Analyse this document image/diagram in detail. "
                    "List every technical term, numeric value, and relationship visible.",
                    compressed,
                ])
                desc = resp.text
            except Exception as e:
                log_event("FAILURE", path.name, "VLM_AGENT", str(e)); continue
            if not ValidationGatekeeper.visual_summary(desc, path.name, img_path.name):
                continue
            chunks.append({
                "text": desc, "source_file": path.name,
                "source_type": "image",
                "page_or_ts": f"image_{img_counter}",
                "image_path": str(img_path),
            })

    log_event("SUCCESS", path.name, "DOCUMENT_PARSER",
              f"Extracted {len(chunks)} chunks.")
    return {"chunks": chunks, "file": path.name}


# ────────────────────────────────────────────────────────────
# TOOL: parse_tabular
# ────────────────────────────────────────────────────────────
def parse_tabular(file_path: str) -> dict:
    """
    Parse a CSV or Excel spreadsheet file.
    Converts each row into a human-readable text chunk
    with column context for semantic embedding.
    """
    path = Path(file_path)
    ext = path.suffix.lower()
    chunks = []
    try:
        df = pd.read_csv(path) if ext == ".csv" else pd.read_excel(path)
    except Exception as e:
        log_event("FAILURE", path.name, "TABULAR_PARSER", str(e))
        return {"chunks": [], "file": path.name}

    header = " | ".join(str(c) for c in df.columns)
    for idx, row in df.iterrows():
        text = f"Table: {path.name}\nColumns: {header}\nRow {idx}: " + \
               " | ".join(str(v) for v in row.values)
        if ValidationGatekeeper.text(text, path.name):
            chunks.append({
                "text": text, "source_file": path.name,
                "source_type": "table",
                "page_or_ts": f"row_{idx}", "image_path": "NONE",
            })
    log_event("SUCCESS", path.name, "TABULAR_PARSER",
              f"Extracted {len(chunks)} chunks.")
    return {"chunks": chunks, "file": path.name}


# ────────────────────────────────────────────────────────────
# TOOL: parse_audio
# ────────────────────────────────────────────────────────────
def parse_audio(file_path: str) -> dict:
    """
    Transcribe an audio file (MP3, WAV, M4A) using OpenAI Whisper
    running locally. No API key required. Returns timestamped
    transcript segments as individual searchable chunks.
    """
    path = Path(file_path)
    chunks = []
    print(f"   [Whisper] Transcribing {path.name}…")
    try:
        result = WHISPER_MODEL.transcribe(str(path), verbose=False)
    except Exception as e:
        log_event("FAILURE", path.name, "AUDIO_PARSER", str(e))
        return {"chunks": [], "file": path.name}

    for seg in result.get("segments", []):
        text = seg.get("text", "").strip()
        start, end = seg.get("start", 0), seg.get("end", 0)
        ts = f"{int(start//60):02d}:{int(start%60):02d}–{int(end//60):02d}:{int(end%60):02d}"
        if ValidationGatekeeper.transcript(text, path.name):
            chunks.append({
                "text": text, "source_file": path.name,
                "source_type": "audio",
                "page_or_ts": ts, "image_path": "NONE",
            })
    log_event("SUCCESS", path.name, "AUDIO_PARSER",
              f"Extracted {len(chunks)} segments.")
    return {"chunks": chunks, "file": path.name}


# ────────────────────────────────────────────────────────────
# TOOL: parse_video
# ────────────────────────────────────────────────────────────
def parse_video(file_path: str, frame_every_n_secs: int = 60) -> dict:
    """
    Process a video file (MP4, MOV) using two parallel tracks:
    Track 1 - Visual: sample one frame every N seconds, describe with VLM.
    Track 2 - Audio: extract with ffmpeg, transcribe with Whisper.
    Returns combined chunks with timestamps from both tracks.
    """
    path = Path(file_path)
    chunks = []

    # Track 1: frames
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 24
    interval = int(fps * frame_every_n_secs)
    fidx = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if fidx % interval == 0:
            ts_secs = fidx / fps
            ts = f"{int(ts_secs//60):02d}:{int(ts_secs%60):02d}"
            pil = PILImage.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            compressed = optimise_image(pil)
            fp_out = FRAME_DIR / f"{path.stem}_frame_{fidx}.png"
            compressed.save(fp_out, "PNG")
            try:
                resp = llm_call([
                    f"Describe this video frame from '{path.name}' at {ts}. "
                    "List every visible object, text on screen, or action.",
                    compressed,
                ])
                desc = resp.text
                if ValidationGatekeeper.visual_summary(desc, path.name, fp_out.name):
                    chunks.append({
                        "text": desc, "source_file": path.name,
                        "source_type": "video_frame",
                        "page_or_ts": f"frame@{ts}", "image_path": str(fp_out),
                    })
            except Exception as e:
                log_event("FAILURE", path.name, "VLM_VIDEO", str(e))
        fidx += 1
    cap.release()

    # Track 2: audio
    audio_out = AUDIO_DIR / f"{path.stem}_audio.wav"
    os.system(f'ffmpeg -y -i "{path}" -vn -ar 16000 -ac 1 "{audio_out}" -loglevel quiet')
    if audio_out.exists():
        audio_result = parse_audio(str(audio_out))
        for c in audio_result["chunks"]:
            c["source_file"] = path.name
            c["source_type"] = "video_audio"
        chunks.extend(audio_result["chunks"])

    log_event("SUCCESS", path.name, "VIDEO_PARSER",
              f"Extracted {len(chunks)} chunks.")
    return {"chunks": chunks, "file": path.name}


# ────────────────────────────────────────────────────────────
# TOOL: parse_image
# ────────────────────────────────────────────────────────────
def parse_image(file_path: str) -> dict:
    """
    Describe a standalone image file (PNG, JPG, WEBP) using
    Gemini Vision. Returns a detailed scene description chunk.
    """
    path = Path(file_path)
    pil = PILImage.open(path)
    compressed = optimise_image(pil)
    try:
        resp = llm_call([
            "Describe this image in detail. List all visible text, "
            "technical terms, relationships, and objects.", compressed])
        desc = resp.text
    except Exception as e:
        log_event("FAILURE", path.name, "IMAGE_PARSER", str(e))
        return {"chunks": [], "file": path.name}
    if not ValidationGatekeeper.visual_summary(desc, path.name, path.name):
        return {"chunks": [], "file": path.name}
    return {"chunks": [{
        "text": desc, "source_file": path.name,
        "source_type": "image", "page_or_ts": "full_image",
        "image_path": str(path),
    }], "file": path.name}


# ────────────────────────────────────────────────────────────
# TOOL: extract_and_index
# ────────────────────────────────────────────────────────────
def extract_and_index(chunks: List[dict], source_file: str) -> dict:
    """
    Take parsed chunks, split into parent/child fragments,
    extract graph triples, and write everything to ChromaDB
    and Neo4j. This is the final indexing step in the pipeline.
    Returns the count of indexed chunks.
    """
    if not chunks:
        return {"indexed": 0, "file": source_file}

    child_col, parent_col = get_chroma_collections()
    indexed = 0

    for chunk in chunks:
        parent_id = "p_" + hashlib.md5(
            (chunk["source_file"] + chunk["text"][:80]).encode()
        ).hexdigest()

        meta = {
            "source_file": chunk["source_file"],
            "source_type": chunk["source_type"],
            "page_or_ts":  chunk["page_or_ts"],
            "image_path":  chunk["image_path"],
        }

        try:
            parent_col.upsert(documents=[chunk["text"]],
                              metadatas=[meta], ids=[parent_id])
        except Exception as e:
            log_event("WARNING", chunk["source_file"], "CHROMA_PARENT", str(e))
            continue

        for i, child in enumerate(chunk_text(chunk["text"])):
            try:
                child_col.upsert(
                    documents=[child],
                    metadatas=[{**meta, "parent_id": parent_id}],
                    ids=[f"c_{parent_id}_{i}"],
                )
            except Exception as e:
                log_event("WARNING", chunk["source_file"], "CHROMA_CHILD", str(e))

        # Graph extraction — skipped if GRAPH_EXTRACTION_ENABLED is False
        if GRAPH_EXTRACTION_ENABLED:
            _write_graph_triples(chunk["text"], chunk["source_file"],
                                 chunk["image_path"])
        indexed += 1

    return {"indexed": indexed, "file": source_file}


def _write_graph_triples(text: str, source_file: str, image_path: str):
    """Internal: extract triples and write to Neo4j. Not an ADK tool."""
    try:
        resp = llm_call(
            "Extract entities and relationships. Return ONLY a valid JSON array. "
            'Each item: {"source": "...", "type": "...", "target": "..."}. '
            f"No markdown.\n\nTEXT:\n{text[:1500]}"
        )
        raw = re.sub(r"^```[a-z]*\n?", "", resp.text.strip()).rstrip("```").strip()
        triples = json.loads(raw)
        if not ValidationGatekeeper.graph_triples(triples, source_file):
            return
    except Exception as e:
        log_event("WARNING", source_file, "GRAPH_EXTRACTOR", str(e))
        return

    if not NEO4J_URI or "YOUR_" in NEO4J_URI:
        return
    try:
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
        with driver.session() as session:
            for t in triples:
                rel = re.sub(r"[^A-Z0-9_]", "", t["type"].upper()) or "RELATED_TO"
                session.run(
                    f"MERGE (s:Entity {{name: $src}}) SET s.source_file=$sf "
                    f"MERGE (t:Entity {{name: $tgt}}) MERGE (s)-[:{rel}]->(t)",
                    src=t["source"], tgt=t["target"], sf=source_file,
                )
        driver.close()
    except Exception as e:
        log_event("WARNING", source_file, "NEO4J_WRITE", str(e))


print("✅ Cell 5 — ADK tool functions (ingestion) ready.")
print("   Tools: parse_document, parse_tabular, parse_audio,")
print("          parse_video, parse_image, extract_and_index")


In [ ]:
# ============================================================
# CELL 6 — ADK TOOL FUNCTIONS (QUERY SIDE)
#
# These are the tools given to the QueryAgent (the LLM-backed
# ADK Agent that handles user questions).
#
# ADK Agent + Tools pattern:
#   The QueryAgent is an LLM Agent. When the user calls ask(),
#   ADK passes the query to the LLM. The LLM decides which
#   tools to call (and in what order) to answer the question.
#   This is different from our old LangGraph approach where WE
#   hardwired the order. ADK lets the LLM reason about it.
#
# However, for predictability we still give it a strong system
# instruction that tells it the mandatory call order:
#   1. search_vector_store  — always first
#   2. search_graph_store   — always second (in parallel mentally)
#   3. generate_answer      — always last, with citation rules
# ============================================================

# ────────────────────────────────────────────────────────────
# TOOL: search_vector_store
# ────────────────────────────────────────────────────────────
def search_vector_store(query: str) -> dict:
    """
    Search the ChromaDB vector knowledge base for content relevant
    to the user's query. Uses parent-document retrieval: finds
    matching child fragments first, then returns their full parent
    passages with source metadata (file name, page, type).
    Always call this tool first for any factual question.
    """
    cache_key = f"qcache_{hashlib.md5(query.encode()).hexdigest()}"
    cached = QUERY_CACHE.get(cache_key)
    if cached:
        print("   [VectorSearch] Cache hit.")
        return {"passages": cached, "count": len(cached)}

    child_col, parent_col = get_chroma_collections()
    if child_col.count() == 0:
        return {"passages": [], "count": 0,
                "warning": "Knowledge base is empty. Ingest files first."}

    child_res = child_col.query(
        query_texts=[query],
        n_results=min(15, child_col.count()),
    )

    seen, parent_ids, image_paths = set(), [], []
    if child_res["metadatas"] and child_res["metadatas"][0]:
        for meta in child_res["metadatas"][0]:
            pid = meta.get("parent_id")
            if pid and pid not in seen:
                parent_ids.append(pid); seen.add(pid)
            ip = meta.get("image_path", "NONE")
            if ip and ip != "NONE" and ip not in image_paths:
                image_paths.append(ip)

    passages = []
    if parent_ids:
        res = parent_col.get(ids=parent_ids[:6])
        for doc, meta in zip(res.get("documents", []),
                             res.get("metadatas", [])):
            passages.append({"text": doc, "metadata": meta})

    QUERY_CACHE.set(cache_key, passages, expire=3600)
    return {"passages": passages, "count": len(passages),
            "image_paths": image_paths}


# ────────────────────────────────────────────────────────────
# TOOL: search_graph_store
# ────────────────────────────────────────────────────────────
def search_graph_store(query: str) -> dict:
    """
    Search the Neo4j knowledge graph for entity relationships
    relevant to the query. Returns relationship paths like
    'ComponentA -[CONNECTS_TO]-> ComponentB (from: manual.pdf)'.
    Call this after search_vector_store to add structural context.
    Returns empty if Neo4j is not configured.
    """
    if not NEO4J_URI or "YOUR_" in NEO4J_URI:
        return {"paths": [], "count": 0}

    paths = []
    try:
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
        with driver.session() as session:
            result = session.run("""
                MATCH (e:Entity)
                WHERE e.name IS NOT NULL
                  AND toLower($q) CONTAINS toLower(e.name)
                MATCH (e)-[r]->(t)
                WHERE t.name IS NOT NULL
                RETURN e.name + ' -[' + type(r) + ']-> ' + t.name +
                       ' (from: ' + COALESCE(e.source_file, 'unknown') + ')'
                       AS path LIMIT 8
            """, q=query)
            paths = [r["path"] for r in result]
        driver.close()
    except Exception as e:
        log_event("WARNING", "query_agent", "GRAPH_SEARCH", str(e))

    return {"paths": paths, "count": len(paths)}


# ────────────────────────────────────────────────────────────
# TOOL: generate_cited_answer
# ────────────────────────────────────────────────────────────
def generate_cited_answer(
    query: str,
    passages: List[dict],
    graph_paths: List[str],
) -> dict:
    """
    Generate a final answer using retrieved passages and graph paths.
    ALWAYS cite retrieved passages as [1], [2], etc.
    ALWAYS label any statement from LLM training data as [LLM knowledge].
    ALWAYS end with a '## Sources used' section.
    If no passages were retrieved, warn the user explicitly.
    Call this tool last, after search_vector_store and search_graph_store.
    """
    global GRAPH_EXTRACTION_ENABLED
    GRAPH_EXTRACTION_ENABLED = False   # Pause graph extraction during query

    try:
        refs, ctx_lines = [], []
        for i, p in enumerate(passages, 1):
            meta  = p.get("metadata", {})
            sf    = meta.get("source_file", "unknown")
            tp    = meta.get("source_type",  "document")
            pt    = meta.get("page_or_ts",   "n/a")
            label = f"[{i}] {sf} | {pt} | {tp}"
            refs.append(label)
            ctx_lines.append(f"{label}\n{p['text']}\n")

        if graph_paths:
            ctx_lines.append("\n[Graph relationships]\n" +
                             "\n".join(graph_paths))

        context = "\n---\n".join(ctx_lines)

        if context.strip():
            user_prompt = (f"USER QUESTION:\n{query}\n\n"
                           f"RETRIEVED CONTEXT:\n{context}")
            system = (
                "You are a precise technical assistant with a document knowledge base.\n\n"
                "MANDATORY RULES — follow all of these exactly:\n"
                "1. Cite retrieved passages inline as [1], [2], etc.\n"
                "2. After any statement from your own training (not in context), "
                "write [LLM knowledge].\n"
                "3. End your response with '## Sources used' listing every "
                "reference you cited, one per line.\n"
                "4. If context contains nothing relevant, begin with: "
                "'⚠️ No relevant context retrieved — answer is from LLM training only.'\n"
                "5. Use Markdown. Be thorough and technical."
            )
        else:
            user_prompt = f"USER QUESTION:\n{query}"
            system = (
                "No document context was retrieved. Begin with: "
                "'⚠️ No relevant context retrieved from the knowledge base. "
                "Answer is from LLM training data only.' Then answer."
            )

        # Stream to console
        print("\n" + "="*64 + "\nANSWER\n" + "="*64 + "\n")
        tokens = []
        stream = _gemini_client.models.generate_content_stream(
            model=TARGET_MODEL,
            contents=user_prompt,
            config=genai_types.GenerateContentConfig(system_instruction=system),
        )
        for chunk in stream:
            t = chunk.text or ""
            print(t, end="", flush=True)
            tokens.append(t)

        answer = "".join(tokens).strip()
        if not answer:
            answer = "❌ Empty response. Please retry."

        if refs and "## Sources used" not in answer:
            sources = "\n\n## Sources used\n" + "\n".join(refs)
            print(sources)
            answer += sources

        print("\n" + "="*64 + "\n")
        return {"answer": answer, "sources": refs}

    finally:
        GRAPH_EXTRACTION_ENABLED = True   # Always re-enable


print("✅ Cell 6 — ADK query tools ready.")
print("   Tools: search_vector_store, search_graph_store, generate_cited_answer")


In [ ]:
# ============================================================
# CELL 7 — ADK AGENTS + WORKFLOW DEFINITION
#
# This is the cell that actually replaces LangGraph's
# StateGraph compile() step.
#
# Two agents are defined:
#
#   ingestion_agent  — not LLM-routed. We call its tools
#                       directly in deterministic order from
#                       our own Python worker loop (Cell 8),
#                       because file ingestion must always run
#                       the same steps in the same order —
#                       there's no judgment call for an LLM
#                       to make here. Wrapping it as an Agent
#                       still gives us ADK's tracing/session
#                       benefits when run via adk web.
#
#   query_agent       — genuinely LLM-routed. The LLM decides
#                       which tools to call. We constrain it
#                       with a strict instruction so it always
#                       calls search_vector_store →
#                       search_graph_store → generate_cited_answer
#                       in that order, but the LLM is reasoning
#                       about it, not us hardwiring edges.
#
# ADK_WORKFLOW_AVAILABLE (set in Cell 3) determines whether we
# also build a graph-based Workflow for the ingestion pipeline.
# If ADK 2.0's Workflow class is present, we use it — it gives
# you the adk web visual debugger for free. If not, we fall
# back to plain function calls (Cell 8 still works either way).
# ============================================================

# ── Query Agent — LLM decides the tool-call order ────────────
query_agent = Agent(
    name="idp_query_agent",
    model=TARGET_MODEL,
    instruction=(
        "You are a retrieval-augmented assistant for a multimodal "
        "document knowledge base containing PDFs, spreadsheets, audio, "
        "video, and images.\n\n"
        "For EVERY user question you MUST follow this exact tool sequence:\n"
        "1. Call search_vector_store with the user's question.\n"
        "2. Call search_graph_store with the user's question.\n"
        "3. Call generate_cited_answer with the question, the passages "
        "from step 1, and the graph paths from step 2.\n"
        "4. Return the 'answer' field from generate_cited_answer verbatim "
        "as your final response — do not paraphrase or summarise it further.\n\n"
        "Never skip search_vector_store, even if you think you know the answer. "
        "Never answer directly without calling generate_cited_answer last."
    ),
    tools=[search_vector_store, search_graph_store, generate_cited_answer],
)

# ── Ingestion Agent — tools called deterministically by us ───
# (LLM routing is unnecessary and risky for an ETL pipeline —
#  we want the SAME steps every time, not LLM judgment calls.)
ingestion_agent = Agent(
    name="idp_ingestion_agent",
    model=TARGET_MODEL,
    instruction=(
        "You are a document ingestion specialist. Given a file path and "
        "its detected type, call the matching parser tool, then call "
        "extract_and_index with the returned chunks to store them."
    ),
    tools=[parse_document, parse_tabular, parse_audio,
           parse_video, parse_image, extract_and_index],
)


# ── Optional: ADK 2.0 graph Workflow for ingestion ───────────
# This gives you a visual, deterministic pipeline you can
# inspect with `adk web` — purely additive, not required for
# the notebook to function (Cell 8's worker calls tools directly).
if ADK_WORKFLOW_AVAILABLE:
    try:
        # Workflow nodes here are illustrative routing agents;
        # the actual heavy lifting still happens inside the
        # parser tool functions themselves.
        router_stub_agent = Agent(
            name="format_router",
            model=TARGET_MODEL,
            instruction=(
                "Identify the file type from its MIME type and state "
                "which parser tool should be used: parse_document, "
                "parse_tabular, parse_audio, parse_video, or parse_image."
            ),
            tools=[],
        )
        ingestion_workflow = Workflow(
            name="ingestion_pipeline",
            edges=[("START", router_stub_agent, ingestion_agent)],
        )
        print("[ADK] ✅ Graph Workflow built (ingestion_pipeline).")
        print("       Run `adk web` in a terminal to visualise it.")
    except Exception as e:
        print(f"[ADK] ⚠️  Workflow construction skipped: {e}")
        ingestion_workflow = None
else:
    ingestion_workflow = None

print("\n✅ Cell 7 — ADK agents defined.")
print("   query_agent      → LLM-routed (used by ask())")
print("   ingestion_agent   → tool container (used by ingestion_worker())")


In [ ]:
# ============================================================
# CELL 8 — ADK RUNNER + FORMAT ROUTER + ASYNC WORKER
#
# ADK difference from LangGraph here:
#   LangGraph: knowledge_agent.invoke(state_dict)
#   ADK:       you need a Runner + Session to invoke an Agent.
#              The Runner manages conversation state (Session)
#              and streams Events back. For our deterministic
#              ingestion path we bypass the Runner entirely and
#              call tool functions directly (see route_and_parse
#              below) — this is faster and avoids LLM token cost
#              for a step that needs no LLM judgment.
#
#              For the QUERY path (Cell 9 / ask()) we DO use the
#              Runner, because that's where we want the LLM to
#              reason about tool order and produce the final text.
# ============================================================

from google.adk.runners import InMemoryRunner

# ── ADK Runner + Session service for the query agent ─────────
# InMemoryRunner bundles a session service for you — good fit
# for a single-notebook MVP. Swap for VertexAiSessionService
# or DatabaseSessionService in production (see Section 9 later).
adk_runner = InMemoryRunner(agent=query_agent, app_name="idp_mvp")

APP_NAME  = "idp_mvp"
USER_ID   = "colab_user"
SESSION_ID = "main_session"

async def _ensure_session():
    """ADK requires an explicit session before running an agent."""
    svc = adk_runner.session_service
    existing = await svc.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )
    if existing is None:
        await svc.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
        )

# Run once at import time (nest_asyncio from Cell 1 allows this in Colab)
import asyncio
asyncio.get_event_loop().run_until_complete(_ensure_session())
print("[ADK] ✅ Session initialised for query_agent.")


# ── Format Router (deterministic — no LLM needed) ────────────
DOCUMENT_MIMES = {
    "application/pdf",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
    "application/vnd.openxmlformats-officedocument.presentationml.presentation",
    "application/msword",
}
TABULAR_MIMES = {
    "text/csv",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    "application/vnd.ms-excel",
}
AUDIO_MIMES = {"audio/mpeg", "audio/wav", "audio/mp4", "audio/ogg", "audio/flac"}
VIDEO_MIMES = {"video/mp4", "video/quicktime", "video/x-msvideo", "video/x-matroska"}
IMAGE_MIMES = {"image/png", "image/jpeg", "image/webp", "image/tiff", "image/gif"}


def route_and_parse(file_path: str) -> dict:
    """
    Detects MIME type and calls the matching ADK tool function
    DIRECTLY (no LLM routing — this is a deterministic ETL step).
    Returns the same {chunks, file} dict every parser tool returns.
    """
    mime = detect_mime(file_path)
    name = Path(file_path).name
    print(f"   [Router] {name} → {mime}")

    if mime in DOCUMENT_MIMES:
        return parse_document(file_path)
    elif mime in TABULAR_MIMES:
        return parse_tabular(file_path)
    elif mime in AUDIO_MIMES:
        return parse_audio(file_path)
    elif mime in VIDEO_MIMES:
        return parse_video(file_path)
    elif mime in IMAGE_MIMES:
        return parse_image(file_path)
    else:
        log_event("WARNING", name, "ROUTER", f"Unsupported MIME '{mime}'.")
        try:
            text = Path(file_path).read_text(errors="replace")
            if ValidationGatekeeper.text(text, name):
                return {"chunks": [{
                    "text": text, "source_file": name,
                    "source_type": "text", "page_or_ts": "full",
                    "image_path": "NONE",
                }], "file": name}
        except Exception:
            pass
        return {"chunks": [], "file": name}


# ── Async ingestion worker (daemon thread) ────────────────────
def ingestion_worker():
    """
    Pulls file paths from INGESTION_QUEUE, runs:
      dedup → route_and_parse() → extract_and_index()
    Both functions are the same ADK tool functions defined in
    Cell 5 — called directly here, not through an LLM, since
    ingestion order is fixed and deterministic.
    """
    print("[Worker] Ingestion worker started.")
    idle = 0
    while True:
        try:
            file_path = INGESTION_QUEUE.get(timeout=5)
            idle = 0
        except queue.Empty:
            idle += 1
            if idle >= 3:
                save_manifest()
                backup_chroma()
                print("[Worker] ✅ Queue empty. Manifest saved. ChromaDB backed up.")
                break
            continue

        path = Path(file_path)
        fhash = file_hash(file_path)

        if fhash in GLOBAL_MANIFEST:
            prev = GLOBAL_MANIFEST[fhash]
            print(f"⏭️  [Skip] '{path.name}' already processed on {prev['processed_at']}")
            INGESTION_QUEUE.task_done()
            continue

        print(f"\n⚡ [Processing] '{path.name}'")
        max_attempts = 2
        for attempt in range(1, max_attempts + 1):
            try:
                parse_result = route_and_parse(file_path)
                index_result = extract_and_index(
                    parse_result["chunks"], parse_result["file"]
                )
                GLOBAL_MANIFEST[fhash] = {
                    "file_name":    path.name,
                    "processed_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
                    "chunks":       index_result["indexed"],
                }
                save_manifest()   # save immediately, not just at queue-empty
                log_event("SUCCESS", path.name, "INGESTION",
                          f"Indexed {index_result['indexed']} chunks.")
                print(f"✅ [Done] '{path.name}' → {index_result['indexed']} chunks indexed.")
                break
            except Exception as e:
                log_event("FAILURE", path.name, "INGESTION",
                          f"Attempt {attempt}/{max_attempts}: {e}")
                if attempt == max_attempts:
                    DEAD_LETTER_QUEUE.append({
                        "file": file_path, "error": str(e),
                        "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
                    })
                    print(f"💀 [Dead Letter] '{path.name}' → {e}")
                else:
                    time.sleep(5)

        INGESTION_QUEUE.task_done()


def submit(file_path: str):
    """Queue a file for ingestion."""
    if not Path(file_path).exists():
        print(f"[Submit] ❌ Not found: {file_path}")
        return
    INGESTION_QUEUE.put(file_path)
    print(f"[Submit] Queued: {Path(file_path).name}")


def start_worker():
    """Start the background ingestion daemon thread."""
    restore_chroma()
    t = threading.Thread(target=ingestion_worker, daemon=True)
    t.start()
    print("[Worker] Background thread started.")
    return t

print("\n✅ Cell 8 — ADK Runner, router, and async worker ready.")


In [ ]:
# ============================================================
# CELL 9 — QUERY EXECUTION VIA ADK RUNNER
#
# This replaces LangGraph's knowledge_agent.invoke(state_dict).
#
# ADK pattern to run an Agent and get its final text response:
#
#   1. Build a genai_types.Content from the user's message
#   2. Call runner.run_async(user_id, session_id, new_message)
#   3. Iterate over the yielded Events
#   4. The LAST event with role="model" and final text IS your answer
#
# ADK's Runner automatically:
#   - Lets the LLM decide which tools to call and in what order
#   - Executes those tool calls (our Cell 6 functions)
#   - Feeds tool results back to the LLM
#   - Streams partial + final response Events
#
# We still print streaming tokens to console for live feedback,
# mirroring the old LangGraph streaming_generator's UX.
# ============================================================

from google.genai import types as genai_types

async def _run_query_async(query: str) -> str:
    """
    Sends the query to query_agent via the ADK Runner and
    returns the final text response. Tool calls happen
    automatically inside this call — search_vector_store,
    search_graph_store, and generate_cited_answer all fire
    as decided by the LLM, per the instruction in Cell 7.
    """
    await _ensure_session()

    user_message = genai_types.Content(
        role="user",
        parts=[genai_types.Part(text=query)],
    )

    final_text = ""
    async for event in adk_runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=user_message,
    ):
        # ADK Events carry partial or final model content.
        # We only care about the final assembled text here —
        # generate_cited_answer already streamed tokens to
        # console itself, so we don't double-print.
        if event.content and event.content.parts:
            for part in event.content.parts:
                if getattr(part, "text", None):
                    final_text = part.text

    return final_text


def ask(query: str):
    """
    Public entry point: run a query through the ADK query_agent.
    The agent autonomously calls search_vector_store →
    search_graph_store → generate_cited_answer (enforced by its
    system instruction in Cell 7), then returns the cited answer.
    """
    print(f"\n{'='*64}\nQUERY: {query}\n{'='*64}")

    try:
        result_text = asyncio.get_event_loop().run_until_complete(
            _run_query_async(query)
        )
    except Exception as e:
        result_text = f"❌ Agent run failed: {e}"
        log_event("FAILURE", "query_agent", "ADK_RUNNER", str(e))

    # Display any matched images inline (Colab)
    try:
        from IPython.display import display
        from PIL import Image as IPImage
        # generate_cited_answer doesn't return image paths directly to
        # this scope, so we re-check the last vector search cache hit.
        # (Optional nicety — safe to skip if no images matched.)
    except Exception:
        pass

    return result_text


print("✅ Cell 9 — ask() wired to ADK Runner. Ready to query.")


In [ ]:
# ============================================================
# CELL 10 — PUBLIC API + UTILITY COMMANDS
# Same public surface as the LangGraph version — only the
# internals (Cells 7–9) changed to use ADK instead of LangGraph.
# ============================================================

import glob as _glob

def start():
    """Boot the background ingestion worker. Call once per session."""
    return start_worker()


def ingest(path_pattern: str):
    """
    Queue a single file, folder, or glob pattern for ingestion.
    Examples:
        ingest("/content/drive/MyDrive/idp_adk/input/manual.pdf")
        ingest("/content/drive/MyDrive/idp_adk/input/")
        ingest("/content/drive/MyDrive/idp_adk/input/*.pdf")
    """
    if "*" in path_pattern or "?" in path_pattern:
        paths = _glob.glob(path_pattern)
    elif Path(path_pattern).is_dir():
        paths = [str(p) for p in Path(path_pattern).iterdir() if p.is_file()]
    else:
        paths = [path_pattern]

    if not paths:
        print(f"[Ingest] No files found matching: {path_pattern}")
        return

    for p in sorted(paths):
        submit(p)
    print(f"[Ingest] Queued {len(paths)} file(s).")


def show_dead_letters():
    """Print files that failed ingestion after all retries."""
    if not DEAD_LETTER_QUEUE:
        print("✅ Dead-letter queue is empty — no failed files.")
        return
    print(f"💀 {len(DEAD_LETTER_QUEUE)} file(s) in dead-letter queue:\n")
    for e in DEAD_LETTER_QUEUE:
        print(f"  File : {e['file']}\n  Error: {e['error']}\n  Time : {e['ts']}\n")


def show_log(n: int = 20):
    """Tail the last N entries from the structured validation log."""
    if not VALIDATION_LOG.exists():
        print("No log file yet.")
        return
    with jsonlines.open(str(VALIDATION_LOG)) as reader:
        entries = list(reader)
    for e in entries[-n:]:
        icon = {"SUCCESS":"✅","FAILURE":"🛑","WARNING":"⚠️","INFO":"ℹ️"}.get(e.get("level",""), "•")
        print(f"{icon} {e.get('ts','')} | {e.get('level','')} | "
              f"{e.get('file','')} | {e.get('type','')} | {e.get('msg','')}")


def show_manifest():
    """List all files that have been successfully ingested."""
    if not GLOBAL_MANIFEST:
        print("No files ingested yet.")
        return
    print(f"{'File':<40} {'Processed At':<22} {'Chunks'}")
    print("-" * 72)
    for v in GLOBAL_MANIFEST.values():
        print(f"{v['file_name']:<40} {v['processed_at']:<22} {v.get('chunks','?')}")


def reset_file(filename_substring: str):
    """
    Remove a file from the manifest so it will be re-ingested
    on the next ingest() call. Useful after a partial failure.
    """
    removed = False
    for h, v in list(GLOBAL_MANIFEST.items()):
        if filename_substring in v.get("file_name", ""):
            del GLOBAL_MANIFEST[h]
            removed = True
    if removed:
        save_manifest()
        print(f"✅ Removed matching file(s) from manifest — ready to re-ingest.")
    else:
        print(f"No manifest entry matched '{filename_substring}'.")


print("""
✅ ADK-based IDP MVP ready. Usage:

   start()                      # Boot ingestion worker (once per session)
   ingest('/path/to/file.pdf')  # Queue a file (or folder / glob)
   ask('your question here')    # Query via ADK query_agent
   show_manifest()              # See ingested files
   show_dead_letters()          # See any failed ingestions
   show_log(30)                 # Tail the validation log
   reset_file('filename')       # Remove from manifest to re-ingest

Architecture note: ingestion still runs deterministically via
direct tool calls (Cell 8). Only the QUERY path (ask()) is
LLM-routed through the ADK Runner + query_agent (Cell 7/9).
""")
